# Мультиагентное RL-управление светофорами: сеть перекрёстков 2×2

**Постановка.** Четыре перекрёстка стоят квадратом 2×2 (СЗ, СВ, ЮЗ, ЮВ),
между соседями есть дороги. На каждом перекрёстке — по одной полосе на
подход, разрешены `straight`, `right` и `left` (разворота нет). Машины
рождаются только на внешних подъездах (по два у каждого узла) и едут по
сети, пока не выедут за её границу.

**Почему это уже интересно.** Одна полоса и три манёвра дают и конфликт
потоков, и блокировку: левый поворот пересекает встречное прямое движение,
поэтому его нельзя пускать вместе с ним, а в общей полосе левая машина на
голове очереди блокирует всех, кто за ней. Агент должен не только выбрать
направление, но и вовремя «вставить» короткую левую фазу.

**Фазы.** 4 защищённые фазы на узел:
1. `0` — С+Ю прямо/направо;
2. `1` — С+Ю лево (защищённая стрелка);
3. `2` — З+В прямо/направо;
4. `3` — З+В лево.

Каждый переход стоит `YELLOW + ALL_RED = 4` секунды потерянного времени,
а полный цикл — 16 секунд, поэтому «дёргать» фазы дорого, и n-step
возвраты в RL здесь особенно важны.

**Состояние (23 числа на агента).** Для каждого из 4 подходов:
`straight/QUEUE_CAPACITY`, `left/QUEUE_CAPACITY`, `right/QUEUE_CAPACITY`,
ожидание головной машины (нормированное, обрезано 1.5). Плюс one-hot
текущей фазы (4), прогресс зелёного `elapsed/MAX_GREEN` и два признака
перехода (флаг и остаток таймера).

**Действия (2).** `switch` — перейти к следующей фазе (порядок фаз
фиксирован, как в реальном контроллере), `stay` — продлить текущую. Маска
не даёт переключаться раньше `MIN_GREEN` и принудительно требует
переключения на `MAX_GREEN`. Фиксированный порядок — принципиальное
решение: при свободном выборе любой фазы агент находил вырожденную
политику «дёргать фазы 0 и 3», морил голодом остальные подходы и упирался
в полный затор.

**Награда (на узел, каждую секунду).**

$$r_t = 1.0\,served_t + 0.3\,right_t - 1.0\,drop_t
  - 0.01\,Q_t - 0.02\,\max_a q_a
  - \min(10^{-4} T^{wait}_t,\ 3) - 0.2\,S_t - 0.5\,switch_t$$

`drop_t` — потерянный спрос (свои прибытия и отвергнутые перетоки),
`Q_t` — суммарная очередь, `max_a q_a` — самая длинная очередь подхода,
`T^{wait}` — суммарное ожидание стоящих машин, `S_t` — число подходов, где
головная машина ждёт дольше `STARVATION_WAIT`.

**Мультиагентность.** 4 агента с parameter sharing: одна сеть на всех,
общий replay buffer, у каждого агента своя n-step очередь. Один шаг среды
даёт 4 перехода — так выборки в буфере получаются разнообразнее, а сеть
учится на состояниях всех узлов сразу.

**Алгоритм.** Double DQN + Dueling-голова + 3-step returns.

Интенсивности подобраны около пропускной способности, а горизонтальная
«магистраль» (подъезды E/W) нагружена в 2.0 раза сильнее вертикальной:
спрос фаз различается, равномерный фиксированный цикл заведомо неоптимален,
при плохой политике появляются отказы (полоса переполнена).

In [ ]:
# ============================================================
# 1. Импорты, топология и гиперпараметры
# ============================================================
import random
import time
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | устройство: {DEVICE}")

# ---------- топология ----------
APPROACHES = ("N", "S", "E", "W")
APPROACH_RU = {"N": "С", "S": "Ю", "E": "В", "W": "З"}
OPPOSITE = {"N": "S", "S": "N", "E": "W", "W": "E"}
# RIGHT_OF[d] / LEFT_OF[d] — сторона справа/слева, если ехать в направлении d
RIGHT_OF = {"N": "E", "E": "S", "S": "W", "W": "N"}
LEFT_OF = {"N": "W", "S": "E", "E": "N", "W": "S"}

# (строка, столбец): 0 — север/запад; NODE_XY — координаты для рисования
NODES = [(0, 0), (0, 1), (1, 0), (1, 1)]
NODE_NAMES = {(0, 0): "СЗ", (0, 1): "СВ", (1, 0): "ЮЗ", (1, 1): "ЮВ"}
NODE_XY = {(0, 0): (0.0, 2.0), (0, 1): (2.0, 2.0),
           (1, 0): (0.0, 0.0), (1, 1): (2.0, 0.0)}
NEIGHBORS = {
    (0, 0): {"E": (0, 1), "S": (1, 0)},
    (0, 1): {"W": (0, 0), "S": (1, 1)},
    (1, 0): {"N": (0, 0), "E": (1, 1)},
    (1, 1): {"N": (0, 1), "W": (1, 0)},
}
# внешние подъезды: машины рождаются только здесь
EXTERNAL = {(0, 0): ("N", "W"), (0, 1): ("N", "E"),
            (1, 0): ("S", "W"), (1, 1): ("S", "E")}


def next_lane(node, approach, maneuver):
    """Куда уходит машина, обслуженная на подходе `approach` узла `node`.

    Возвращает (соседний узел, подход соседа) или None, если машина
    покидает сеть. Переток применяется в конце шага, поэтому за секунду
    машина пересекает не больше одного перекрёстка.
    """
    travel = OPPOSITE[approach]
    if maneuver == "straight":
        side = travel
    elif maneuver == "right":
        side = RIGHT_OF[travel]
    else:  # left
        side = LEFT_OF[travel]
    nb = NEIGHBORS[node].get(side)
    return None if nb is None else (nb, OPPOSITE[side])


# ---------- потоки ----------
DT = 1.0                        # секунд на шаг решения
ARRIVAL_RATE = 0.07             # базовый поток, авто/с на внешний подъезд
# горизонтальная «магистраль» загружена сильнее: спрос фаз различается
RATE_WEIGHT = {"N": 1.0, "S": 1.0, "E": 2.0, "W": 2.0}
MANEUVERS = ("left", "straight", "right")
MANEUVER_P = {"left": 0.12, "straight": 0.63, "right": 0.25}
SERVICE_TIME = {"straight": 2.0, "right": 1.8, "left": 2.4}
QUEUE_CAPACITY = 30             # авто на полосу
MAX_WAIT_NORM = 180.0           # нормировка ожиданий в состоянии

# ---------- фазы ----------
PHASES = (0, 1, 2, 3)
N_PHASES = len(PHASES)
PHASE_ACTIVE = {0: ("N", "S"), 1: ("N", "S"), 2: ("E", "W"), 3: ("E", "W")}
PHASE_MANEUVERS = {0: ("straight", "right"), 1: ("left",),
                   2: ("straight", "right"), 3: ("left",)}
PHASE_NAMES = {0: "С-Ю прямо", 1: "С-Ю лево",
               2: "З-В прямо", 3: "З-В лево"}
PHASE_SHORT = {0: "С-Ю ▲", 1: "С-Ю ←", 2: "З-В ▲", 3: "З-В ←"}

MIN_GREEN = 6      # минимальная длительность зелёного, с
MAX_GREEN = 40      # после неё переключение обязательно
YELLOW_TIME = 3
ALL_RED_TIME = 1

ACTION_SWITCH = 0
ACTION_STAY = 1
N_ACTIONS = 2

# ---------- награда ----------
W_THROUGHPUT = 1.0
W_RIGHT_BONUS = 0.3
W_DROP = 1.0
W_QUEUE = 0.01
W_IMBALANCE = 0.02
W_TOTAL_WAIT = 1e-4
TOTAL_WAIT_CAP = 3.0
W_STARVATION = 0.2
W_SWITCH = 0.5
STARVATION_WAIT = 90.0

# ---------- эпизод и RL ----------
EPISODE_STEPS = 900     # 15 минут модельного времени
N_EPISODES = 300

GAMMA = 0.99
LR = 3e-4
BATCH_SIZE = 128
BUFFER_SIZE = 100_000
WARMUP_STEPS = 2_000
TRAIN_EVERY = 2
TARGET_UPDATE = 1_000   # в градиентных шагах
GRAD_CLIP = 10.0
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 150_000
HIDDEN = 128
N_STEP = 3

EVAL_EPISODES = 3
FIXED_GREEN = 12
MODEL_PATH = "lesson_traffic_lite_dqn_2x2.pt"

# 4 подхода * (straight + left + right + ожидание) + фазы + 3 служебных
STATE_DIM = len(APPROACHES) * 4 + N_PHASES + 3

# ---------- оформление ----------
ARM = 1.00          # длина плеча дороги (равна половине шага сетки)
ROAD_W = 0.26       # ширина полосы
LANE_OFF = 0.18     # боковой сдвиг входящей полосы
ROAD_W_TOTAL = 2 * LANE_OFF + ROAD_W   # ширина дороги с двумя полосами
NODE_S = 0.50       # сторона квадрата перекрёстка
CAR_L = 0.14
CAR_W = 0.09
DIRVEC = {"N": (0.0, 1.0), "S": (0.0, -1.0),
          "E": (1.0, 0.0), "W": (-1.0, 0.0)}
C_GREEN, C_RED, C_YELLOW = "#27ae60", "#e74c3c", "#f39c12"
C_ROAD, C_NODE, C_CAR, C_RIGHTCAR, C_LEFTCAR, C_EDGE = (
    "#c8d0d8", "#ffffff", "#2c3e50", "#e67e22", "#8e44ad", "#34495e")

print(f"state_dim={STATE_DIM}, n_actions={N_ACTIONS}, n_step={N_STEP}, "
      f"агентов={len(NODES)}")

In [ ]:
# ============================================================
# 2. Окружение: сеть 2×2
# ============================================================
class Vehicle:
    __slots__ = ("maneuver", "arrival_step")

    def __init__(self, maneuver, arrival_step):
        self.maneuver = maneuver
        self.arrival_step = arrival_step


class Intersection:
    """Один перекрёсток: 4 защищённые фазы, по одной полосе на подход.

    В полосе машины стоят в общем порядке, поэтому левая машина на голове
    блокирует прямо/направо: в левой фазе разгружаются только `left`,
    в фазе прямо/направо — только `straight`/`right` с головы очереди.

    Счётчики по манёврам и сумма шагов прибытия поддерживаются
    инкрементально: без этого подсчёт очередей на каждом шаге (4 узла ×
    4 подхода × до 30 машин) съедал бы больше времени, чем сам шаг RL.
    """

    def __init__(self, node):
        self.node = node
        self.external = EXTERNAL[node]
        self.reset()

    def reset(self):
        self.phase = 0
        self.mode = "green"          # green | yellow | all_red
        self.green_elapsed = 0
        self.timer = 0
        self.pending_phase = None
        self.queues = {a: deque() for a in APPROACHES}
        self.lane_acc = {a: 0.0 for a in APPROACHES}
        self.n_straight = {a: 0 for a in APPROACHES}
        self.n_left = {a: 0 for a in APPROACHES}
        self.n_right = {a: 0 for a in APPROACHES}
        self.arrival_sum = {a: 0.0 for a in APPROACHES}
        self.served = 0
        self.served_right = 0
        self.dropped = 0
        self.switched = 0
        self.served_wait_sum = 0.0
        self.starving_steps = 0

    # ---------- маска действий ----------
    def action_mask(self):
        """Порядок фаз фиксирован, агент решает только «когда переключать».

        Как и в реальном контроллере: следующая фаза — заданная
        последовательность. Пропускать фазы нельзя, иначе агент находит
        вырожденные стратегии вида «дёргать 0 и 3» и морит голодом
        остальные подходы.
        """
        mask = np.zeros(N_ACTIONS, dtype=bool)
        mask[ACTION_STAY] = True
        if self.mode == "green":
            mask[ACTION_SWITCH] = self.green_elapsed >= MIN_GREEN
            if self.green_elapsed >= MAX_GREEN:
                mask[ACTION_STAY] = False       # переключение обязательно
        return mask

    # ---------- переключение фаз ----------
    def apply_action(self, action):
        action = int(action)
        if not self.action_mask()[action]:
            action = ACTION_STAY
        switched = 0
        if self.mode == "green":
            if action == ACTION_SWITCH or self.green_elapsed >= MAX_GREEN:
                self._start_yellow((self.phase + 1) % N_PHASES)
                switched = 1
        self.switched += switched
        return switched

    def _start_yellow(self, pending):
        self.mode = "yellow"
        self.timer = YELLOW_TIME
        self.pending_phase = pending

    def tick(self):
        if self.mode == "green":
            self.green_elapsed += 1
        elif self.mode == "yellow":
            self.timer -= 1
            if self.timer <= 0:
                self.mode = "all_red"
                self.timer = ALL_RED_TIME
        else:  # all_red
            self.timer -= 1
            if self.timer <= 0:
                self.mode = "green"
                self.phase = self.pending_phase
                self.pending_phase = None
                self.green_elapsed = 0

    # ---------- очередь ----------
    def enqueue(self, approach, vehicle):
        if self.total_at(approach) >= QUEUE_CAPACITY:
            return False
        self.queues[approach].append(vehicle)
        self.arrival_sum[approach] += vehicle.arrival_step
        if vehicle.maneuver == "right":
            self.n_right[approach] += 1
        elif vehicle.maneuver == "left":
            self.n_left[approach] += 1
        else:
            self.n_straight[approach] += 1
        return True

    def total_at(self, approach):
        return (self.n_straight[approach] + self.n_left[approach]
                + self.n_right[approach])

    def total_queue(self):
        return sum(self.total_at(a) for a in APPROACHES)

    def discharge(self, t):
        """Разгрузка головы активных подходов; возвращает (подход, машина)."""
        out = []
        if self.mode != "green":
            return out
        allowed = PHASE_MANEUVERS[self.phase]
        for a in PHASE_ACTIVE[self.phase]:
            q = self.queues[a]
            if not q:
                self.lane_acc[a] = 0.0
                continue
            if q[0].maneuver not in allowed:
                self.lane_acc[a] = 0.0      # голова заблокирована
                continue
            acc = self.lane_acc[a] + DT / SERVICE_TIME[q[0].maneuver]
            if acc >= 1.0:
                v = q.popleft()
                acc -= 1.0
                self.arrival_sum[a] -= v.arrival_step
                if v.maneuver == "right":
                    self.n_right[a] -= 1
                    self.served_right += 1
                elif v.maneuver == "left":
                    self.n_left[a] -= 1
                else:
                    self.n_straight[a] -= 1
                self.served += 1
                self.served_wait_sum += (t - v.arrival_step) * DT
                out.append((a, v))
            self.lane_acc[a] = acc if q else 0.0
        return out

    # ---------- метрики узла ----------
    def metrics(self, t):
        total_wait, starving, max_q = 0.0, 0, 0
        for a in APPROACHES:
            n = self.total_at(a)
            if n == 0:
                continue
            if n > max_q:
                max_q = n
            total_wait += (n * t - self.arrival_sum[a]) * DT
            if (t - self.queues[a][0].arrival_step) * DT > STARVATION_WAIT:
                starving += 1
        return {"queue": self.total_queue(), "max_q": max_q,
                "total_wait": total_wait, "starving": starving}

    # ---------- state ----------
    def state(self, t):
        s = np.zeros(STATE_DIM, dtype=np.float32)
        i = 0
        for a in APPROACHES:
            q = self.queues[a]
            s[i + 0] = self.n_straight[a] / QUEUE_CAPACITY
            s[i + 1] = self.n_left[a] / QUEUE_CAPACITY
            s[i + 2] = self.n_right[a] / QUEUE_CAPACITY
            if q:
                wait = (t - q[0].arrival_step) * DT
                s[i + 3] = min(wait / MAX_WAIT_NORM, 1.5)
            i += 4
        s[i + self.phase] = 1.0
        i += N_PHASES
        s[i + 0] = min(self.green_elapsed / MAX_GREEN, 1.0)
        s[i + 1] = 0.0 if self.mode == "green" else 1.0
        s[i + 2] = (self.timer / (YELLOW_TIME + ALL_RED_TIME)
                    if self.mode != "green" else 0.0)
        return s


class TrafficNetworkEnv:
    """Сеть из 4 перекрёстков; интерфейс — словари по узлам.

    Шаг: решения -> прибытия -> разгрузка -> перетоки -> награда -> таймеры.
    Перетоки применяются в конце шага, поэтому машина не проезжает два
    перекрёстка за одну секунду. Если у получателя полоса переполнена,
    отказ засчитывается принимающему агенту.
    """

    def __init__(self, seed=None, episode_steps=EPISODE_STEPS):
        self.rng = np.random.default_rng(seed)
        self.episode_steps = episode_steps
        self.reset()

    def reset(self):
        self.t = 0
        self.nodes = {n: Intersection(n) for n in NODES}
        self.arrived = 0
        self.completed = 0          # выехали за границу сети
        self.dropped_total = 0      # не влезли в полосу
        return self.state()

    def state(self):
        return {n: nd.state(self.t) for n, nd in self.nodes.items()}

    def action_masks(self):
        return {n: nd.action_mask() for n, nd in self.nodes.items()}

    def in_network(self):
        return sum(nd.total_queue() for nd in self.nodes.values())

    def _spawn(self):
        drops = {n: 0 for n in NODES}
        for n, nd in self.nodes.items():
            for a in nd.external:
                rate = ARRIVAL_RATE * RATE_WEIGHT[a]
                for _ in range(self.rng.poisson(rate * DT)):
                    self.arrived += 1
                    m = str(self.rng.choice(MANEUVERS,
                                            p=[MANEUVER_P[k] for k in MANEUVERS]))
                    if not nd.enqueue(a, Vehicle(m, self.t)):
                        nd.dropped += 1
                        self.dropped_total += 1
                        drops[n] += 1
        return drops

    def step(self, actions):
        switched = {n: nd.apply_action(actions[n])
                    for n, nd in self.nodes.items()}
        drops = self._spawn()

        served_now = {n: 0 for n in NODES}
        served_right = {n: 0 for n in NODES}
        transfers = []
        for n, nd in self.nodes.items():
            for a, v in nd.discharge(self.t):
                served_now[n] += 1
                if v.maneuver == "right":
                    served_right[n] += 1
                transfers.append((n, a, v))

        # перетоки — в конце шага
        for n, a, v in transfers:
            dest = next_lane(n, a, v.maneuver)
            if dest is None:
                self.completed += 1
                continue
            nb, na = dest
            if not self.nodes[nb].enqueue(na, v):
                self.nodes[nb].dropped += 1
                self.dropped_total += 1
                drops[nb] += 1

        rewards, infos = {}, {}
        for n, nd in self.nodes.items():
            met = nd.metrics(self.t)
            wait_penalty = min(W_TOTAL_WAIT * met["total_wait"], TOTAL_WAIT_CAP)
            rewards[n] = float(
                W_THROUGHPUT * served_now[n]
                + W_RIGHT_BONUS * served_right[n]
                - W_DROP * drops[n]
                - W_QUEUE * met["queue"]
                - W_IMBALANCE * met["max_q"]
                - wait_penalty
                - W_STARVATION * met["starving"]
                - W_SWITCH * switched[n]
            )
            infos[n] = {
                "served_now": served_now[n],
                "served_right": served_right[n],
                "dropped_now": drops[n],
                "queue": met["queue"],
                "max_q": met["max_q"],
                "total_wait": met["total_wait"],
                "starving": met["starving"],
                "switched": switched[n],
                "phase": nd.phase,
                "mode": nd.mode,
            }

        for nd in self.nodes.values():
            nd.tick()
        self.t += 1
        done = self.t >= self.episode_steps
        return self.state(), rewards, done, infos

    # ---------- снимок для визуализации ----------
    def snapshot(self):
        snap = {"t": self.t, "completed": self.completed,
                "dropped": self.dropped_total, "arrived": self.arrived,
                "in_network": self.in_network(), "nodes": {}}
        for n, nd in self.nodes.items():
            # первые 7 машин каждой полосы в порядке очереди — для отрисовки
            seq = {a: [v.maneuver for v in nd.queues[a]][:7]
                   for a in APPROACHES}
            counts = {a: nd.total_at(a) for a in APPROACHES}
            snap["nodes"][n] = {
                "phase": nd.phase, "mode": nd.mode,
                "green": nd.green_elapsed, "seq": seq, "counts": counts,
                "served": nd.served, "dropped": nd.dropped,
            }
        return snap

In [ ]:
# ============================================================
# 3. Отрисовка сети (matplotlib)
# ============================================================
def _arm_xy(xy, a, d, off=LANE_OFF):
    """Точка на плече `a` узла `xy` на расстоянии d от центра."""
    x, y = xy
    dx, dy = DIRVEC[a]
    lx, ly = DIRVEC[RIGHT_OF[OPPOSITE[a]]]
    return x + dx * d + lx * off, y + dy * d + ly * off


def _road_rect(ax, xy, a, z=1):
    cx, cy = _arm_xy(xy, a, ARM / 2, off=0.0)
    w, h = ((ROAD_W_TOTAL, ARM) if a in ("N", "S")
            else (ARM, ROAD_W_TOTAL))
    ax.add_patch(Rectangle((cx - w / 2, cy - h / 2), w, h,
                           facecolor=C_ROAD, edgecolor="none", zorder=z))


def _center_line(ax, xy, a, z=2):
    x0, y0 = _arm_xy(xy, a, NODE_S / 2 + 0.06, off=0.0)
    x1, y1 = _arm_xy(xy, a, ARM, off=0.0)
    ax.plot([x0, x1], [y0, y1], color="#e8eef3", lw=0.8,
            linestyle=(0, (4, 6)), zorder=z)


def _car_rect(ax, xy, a, d, length, color, z=6):
    cx, cy = _arm_xy(xy, a, d)
    w, h = (CAR_W, length) if a in ("N", "S") else (length, CAR_W)
    ax.add_patch(Rectangle((cx - w / 2, cy - h / 2), w, h,
                           facecolor=color, edgecolor="white",
                           linewidth=0.5, zorder=z))


def _stop_line(ax, xy, a, color, z=7):
    """Стоп-линия поперёк дороги; её цвет и есть сигнал светофора."""
    cx, cy = _arm_xy(xy, a, 0.34)
    if a in ("N", "S"):
        xs, ys = [cx - ROAD_W_TOTAL / 2, cx + ROAD_W_TOTAL / 2], [cy, cy]
    else:
        xs, ys = [cx, cx], [cy - ROAD_W_TOTAL / 2, cy + ROAD_W_TOTAL / 2]
    ax.plot(xs, ys, color=color, lw=3.4, solid_capstyle="butt", zorder=z)


def _signal_color(node_snap, a):
    if node_snap["mode"] == "all_red":
        return C_RED
    active = PHASE_ACTIVE[node_snap["phase"]]
    if node_snap["mode"] == "yellow":
        return C_YELLOW if a in active else C_RED
    return C_GREEN if a in active else C_RED


def draw_snapshot(snap, ax=None, legend=True):
    """Рисует один кадр по словарю из `env.snapshot()`."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7.4, 7.6))
    ax.set_facecolor("#f7f9fb")

    for xy in NODE_XY.values():
        for a in APPROACHES:
            _road_rect(ax, xy, a)
            _center_line(ax, xy, a)

    for n, nd in snap["nodes"].items():
        xy = NODE_XY[n]
        for a in APPROACHES:
            _stop_line(ax, xy, a, _signal_color(nd, a))
            seq = nd["seq"][a]
            if seq:
                step = 0.50 / len(seq)
                length = min(CAR_L, step * 0.85)
                for k, man in enumerate(seq):
                    # машины стоят от стоп-линии наружу
                    color = {"left": C_LEFTCAR, "right": C_RIGHTCAR}.get(
                        man, C_CAR)
                    _car_rect(ax, xy, a, 0.40 + step * (k + 0.5), length,
                              color)
        x, y = xy
        ax.add_patch(Rectangle((x - NODE_S / 2, y - NODE_S / 2),
                               NODE_S, NODE_S, facecolor=C_NODE,
                               edgecolor=C_EDGE, linewidth=1.4, zorder=8))
        ax.text(x, y + 0.12, NODE_NAMES[n], ha="center", va="center",
                fontsize=9, fontweight="bold", color=C_EDGE, zorder=9)
        ax.text(x, y - 0.02, PHASE_SHORT[nd["phase"]], ha="center",
                va="center", fontsize=7.5, color="#7f8c8d", zorder=9)
        ax.text(x, y - 0.16, f"q={sum(nd['counts'].values())}", ha="center",
                va="center", fontsize=6.5, color="#7f8c8d", zorder=9)

    ax.set_xlim(-1.25, 3.25)
    ax.set_ylim(-1.25, 3.25)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"t = {snap['t']} с   |   в сети: {snap['in_network']}   |   "
                 f"выехало: {snap['completed']}   |   отказы: {snap['dropped']}",
                 fontsize=10)

    if legend:
        handles = [
            Rectangle((0, 0), 1, 1, facecolor=C_ROAD, label="дорога"),
            Rectangle((0, 0), 1, 1, facecolor=C_CAR, label="прямо"),
            Rectangle((0, 0), 1, 1, facecolor=C_RIGHTCAR, label="направо (ППД)"),
            Rectangle((0, 0), 1, 1, facecolor=C_LEFTCAR, label="налево"),
            plt.Line2D([0], [0], color=C_GREEN, lw=3.4, label="зелёный"),
            plt.Line2D([0], [0], color=C_YELLOW, lw=3.4, label="жёлтый"),
            plt.Line2D([0], [0], color=C_RED, lw=3.4, label="красный"),
        ]
        ax.legend(handles=handles, loc="lower center",
                  bbox_to_anchor=(0.5, -0.02), ncol=6,
                  frameon=False, fontsize=7.5, handlelength=1.2,
                  columnspacing=1.0)
    return ax


def show_network(env, legend=True):
    fig, ax = plt.subplots(figsize=(7.4, 7.8))
    draw_snapshot(env.snapshot(), ax=ax, legend=legend)
    fig.tight_layout()
    plt.show()


def ascii_summary(env):
    """Компактная текстовая сводка: очереди по подходам и ожидания."""
    print(f"t={env.t:4d}с | в сети={env.in_network():3d} | "
          f"выехало={env.completed:4d} | отказы={env.dropped_total:3d} | "
          f"прибыло={env.arrived:4d}")
    head = f"{'узел':<6}{'фаза':<12}{'сигнал':<12}"
    head += "".join(f"{APPROACH_RU[a]:>5}" for a in APPROACHES)
    head += f"{'макс.ожид':>11}{'обслужено':>11}"
    print(head)
    for n, nd in env.nodes.items():
        sig = {"green": "зелёный", "yellow": "жёлтый",
               "all_red": "все-красный"}[nd.mode]
        waits = [((env.t - nd.queues[a][0].arrival_step) * DT) if nd.queues[a]
                 else 0.0 for a in APPROACHES]
        row = f"{NODE_NAMES[n]:<6}{PHASE_NAMES[nd.phase]:<12}{sig:<12}"
        row += "".join(f"{nd.total_at(a):>5d}" for a in APPROACHES)
        row += f"{max(waits):>10.0f}с{nd.served:>11d}"
        print(row)

In [ ]:
# ============================================================
# 4. Проверка окружения: 60 шагов и статичная картинка
# ============================================================
env = TrafficNetworkEnv(seed=SEED, episode_steps=200)
states = env.reset()
for i in range(60):
    actions = {n: ACTION_STAY for n in NODES}
    if i == 25:                # после MIN_GREEN сдвигаем фазу у всех узлов
        actions = {n: ACTION_SWITCH for n in NODES}
    elif i == 50:              # восточные узлы просят следующий шаг цикла
        actions = {n: (ACTION_SWITCH if n in ((0, 1), (1, 1))
                       else ACTION_STAY) for n in NODES}
    states, rewards, done, infos = env.step(actions)

print("состояние одного агента (23 числа):")
print(np.round(states[NODES[0]], 2))
print(f"награда за последний шаг по узлам: "
      f"{ {NODE_NAMES[n]: round(rewards[n], 2) for n in NODES} }")
ascii_summary(env)

assert all(s.shape == (STATE_DIM,) and np.isfinite(s).all()
           for s in states.values())
assert env.arrived == env.completed + env.dropped_total + env.in_network()
print("\nинвариант сохранения машин выполнен: "
      "прибыло = выехало + отказы + в сети")
show_network(env)

## 5. Нейросеть

Dueling MLP: backbone 23 -> 128 -> 128, затем две головы — ценность
состояния V(s) и преимущества действий A(s, a); Q = V + A − mean(A).
Одна сеть обслуживает все 4 узла (parameter sharing): вход — состояние
конкретного узла, поэтому «сеть перекрёстка» не зависит от его позиции.

In [ ]:
# ============================================================
# 5. Нейросеть (Dueling DQN)
# ============================================================
class DQNNet(nn.Module):
    def __init__(self, state_dim=STATE_DIM, n_actions=N_ACTIONS,
                 hidden=HIDDEN):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.value = nn.Linear(hidden, 1)
        self.advantage = nn.Linear(hidden, n_actions)

    def forward(self, x):
        f = self.feature(x)
        v = self.value(f)
        a = self.advantage(f)
        return v + a - a.mean(dim=1, keepdim=True)


_net_test = DQNNet()
n_params = sum(p.numel() for p in _net_test.parameters())
print(_net_test)
print(f"параметров: {n_params:,} | выход: "
      f"{tuple(_net_test(torch.zeros(2, STATE_DIM)).shape)}")

## 6. Replay buffer и n-step обёртка

`ReplayBuffer` — кольцевой буфер на numpy; рядом с `s, a, r, s'` храним
`next_mask`, чтобы при расчёте target не выбирать недопустимые действия.
`NStepBuffer` — per-agent обёртка: копит `n` переходов **своего** агента и
кладёт в **общий** буфер агрегат `(s_t, a_t, R_n, s_{t+n})`,
`R_n = Σ γ^i r_{t+i}`. Общий буфер на 4 агента даёт вчетверо больше
переходов за шаг среды и перемешивает состояния разных узлов.

In [ ]:
# ============================================================
# 6. Replay buffer (общий) и n-step (на агента)
# ============================================================
class ReplayBuffer:
    def __init__(self, capacity=BUFFER_SIZE, state_dim=STATE_DIM,
                 n_actions=N_ACTIONS):
        self.capacity = capacity
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)
        self.next_masks = np.zeros((capacity, n_actions), dtype=bool)
        self.ptr = 0
        self.size = 0

    def push(self, s, a, r, s2, done, next_mask):
        i = self.ptr
        self.states[i] = s
        self.actions[i] = a
        self.rewards[i] = r
        self.next_states[i] = s2
        self.dones[i] = float(done)
        self.next_masks[i] = next_mask
        self.ptr = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        idx = rng.integers(0, self.size, size=batch_size)
        return {
            "states": torch.as_tensor(self.states[idx], device=DEVICE),
            "actions": torch.as_tensor(self.actions[idx], device=DEVICE),
            "rewards": torch.as_tensor(self.rewards[idx], device=DEVICE),
            "next_states": torch.as_tensor(self.next_states[idx], device=DEVICE),
            "dones": torch.as_tensor(self.dones[idx], device=DEVICE),
            "next_masks": torch.as_tensor(self.next_masks[idx], device=DEVICE),
        }

    def __len__(self):
        return self.size


class NStepBuffer:
    """n-step возвраты одного агента поверх общего ReplayBuffer."""

    def __init__(self, inner, n=N_STEP, gamma=GAMMA):
        self.inner = inner
        self.n = n
        self.gamma = gamma
        self.pending = deque()

    def _emit(self):
        n = min(self.n, len(self.pending))
        R, disc, done = 0.0, 1.0, 0.0
        for i in range(n):
            r, d = self.pending[i][2], self.pending[i][4]
            R += disc * r
            disc *= self.gamma
            if d:
                done = 1.0
                n = i + 1        # не заглядываем за конец эпизода
                break
        s, a = self.pending[0][0], self.pending[0][1]
        last = self.pending[n - 1]
        self.inner.push(s, a, R, last[3], done, last[5])
        self.pending.popleft()

    def push(self, s, a, r, s2, done, next_mask):
        self.pending.append((s, a, r, s2, float(done), next_mask))
        if len(self.pending) >= self.n:
            self._emit()
        if done:                 # хвост эпизода тоже должен попасть в буфер
            while self.pending:
                self._emit()

    def sample(self, batch_size, rng):
        return self.inner.sample(batch_size, rng)

    def __len__(self):
        return len(self.inner)

## 7. Выбор действий и loss

* `select_actions` — ε-greedy **только по допустимым действиям**, сразу для
  всех 4 агентов: состояния складываются в батч `(4, 17)`, один forward;
* `compute_loss` — Double DQN: действие выбирает online-сеть, оценивает
  target-сеть, loss — Huber (`smooth_l1`).

In [ ]:
# ============================================================
# 7. Выбор действий и loss
# ============================================================
def epsilon_by_step(step):
    if step >= EPS_DECAY_STEPS:
        return EPS_END
    return EPS_START + (EPS_END - EPS_START) * step / EPS_DECAY_STEPS


def select_actions(net, states, masks, epsilon, rng):
    """ε-greedy для всех агентов сразу; masks — bool (n_agents, n_actions)."""
    with torch.no_grad():
        x = torch.as_tensor(np.asarray(states), dtype=torch.float32,
                            device=DEVICE)
        q = net(x).cpu().numpy()
    q = np.where(masks, q, -np.inf)
    actions = q.argmax(axis=1).astype(np.int64)
    if epsilon > 0.0:
        explore = np.flatnonzero(rng.random(len(actions)) < epsilon)
        for i in explore:
            actions[i] = int(rng.choice(np.flatnonzero(masks[i])))
    return actions


def compute_loss(online_net, target_net, batch, gamma=GAMMA):
    """Double DQN: действие выбирает online-сеть, оценивает target-сеть."""
    q = online_net(batch["states"]).gather(
        1, batch["actions"].unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_q = online_net(batch["next_states"]).masked_fill(
            ~batch["next_masks"], -1e9)
        next_actions = next_q.argmax(dim=1, keepdim=True)
        next_q_target = target_net(batch["next_states"]).gather(
            1, next_actions).squeeze(1)
        y = batch["rewards"] + gamma * next_q_target * (1.0 - batch["dones"])
    return F.smooth_l1_loss(q, y)

## 8. Цикл обучения

Один шаг: для всех узлов ε-greedy -> `env.step` -> по одному переходу
каждого агента в его n-step буфер -> (каждые `TRAIN_EVERY` шагов) градиент
-> (каждые `TARGET_UPDATE` градиентных шагов) синхронизация target-сети.
ε линейно убывает от `EPS_START` до `EPS_END` за `EPS_DECAY_STEPS` шагов.
В лог добавлены served по узлам, выехавшие машины и число переключений —
по ним сразу видно вырождение политики (например, когда один узел «держит»
зелёный и морит остальные подходы).

In [ ]:
# ============================================================
# 8. Обучение
# ============================================================
def train_agent(episodes=N_EPISODES, seed=SEED, verbose=True,
                episode_steps=EPISODE_STEPS):
    rng = np.random.default_rng(seed)
    online = DQNNet().to(DEVICE)
    target = DQNNet().to(DEVICE)
    target.load_state_dict(online.state_dict())
    target.eval()
    optim = torch.optim.Adam(online.parameters(), lr=LR)
    replay = ReplayBuffer()
    bufs = [NStepBuffer(replay) for _ in NODES]
    env = TrafficNetworkEnv(seed=seed * 1000 + 1,
                            episode_steps=episode_steps)

    hist = {"reward": [], "queue": [], "served": [], "completed": [],
            "wait": [], "starvation": [], "loss": [], "epsilon": [],
            "switches": []}
    losses = deque(maxlen=500)
    global_step, grad_steps = 0, 0
    t0 = time.time()

    for ep in range(1, episodes + 1):
        states = env.reset()
        ep_r, q_sum, starv, switches = 0.0, 0.0, 0.0, 0
        while True:
            masks = env.action_masks()
            arr = np.stack([states[n] for n in NODES])
            msk = np.stack([masks[n] for n in NODES])
            acts = select_actions(online, arr, msk,
                                  epsilon_by_step(global_step), rng)
            actions = {n: int(acts[i]) for i, n in enumerate(NODES)}

            next_states, rewards, done, infos = env.step(actions)
            next_masks = env.action_masks()
            for i, n in enumerate(NODES):
                bufs[i].push(states[n], actions[n], rewards[n],
                             next_states[n], done, next_masks[n])
            states = next_states

            ep_r += float(sum(rewards.values()))
            q_sum += sum(info["queue"] for info in infos.values())
            starv += (sum(1 for info in infos.values() if info["starving"] > 0)
                      / len(NODES))
            switches += sum(info["switched"] for info in infos.values())
            global_step += 1

            if len(replay) >= WARMUP_STEPS and global_step % TRAIN_EVERY == 0:
                batch = replay.sample(BATCH_SIZE, rng)
                loss = compute_loss(online, target, batch)
                optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(online.parameters(), GRAD_CLIP)
                optim.step()
                losses.append(loss.item())
                grad_steps += 1
                if grad_steps % TARGET_UPDATE == 0:
                    target.load_state_dict(online.state_dict())

            if done:
                break

        served_total = sum(nd.served for nd in env.nodes.values())
        wait_sum = sum(nd.served_wait_sum for nd in env.nodes.values())
        hist["reward"].append(ep_r)
        hist["queue"].append(q_sum / env.t)
        hist["served"].append(served_total)
        hist["completed"].append(env.completed)
        hist["wait"].append(wait_sum / max(1, served_total))
        hist["starvation"].append(starv / env.t)
        hist["loss"].append(float(np.mean(losses)) if losses else 0.0)
        hist["epsilon"].append(epsilon_by_step(global_step))
        hist["switches"].append(switches)

        if verbose and (ep == 1 or ep % 10 == 0):
            srv = "/".join(str(env.nodes[n].served) for n in NODES)
            print(f"эп {ep:3d} | R={ep_r:9.1f} | очередь={hist['queue'][-1]:5.1f} | "
                  f"served={served_total:4d} (СЗ/СВ/ЮЗ/ЮВ={srv}) | "
                  f"выехало={env.completed:4d} | ожид={hist['wait'][-1]:5.1f}с | "
                  f"starv={hist['starvation'][-1] * 100:4.1f}% | sw={switches:3d} | "
                  f"eps={hist['epsilon'][-1]:.3f} | loss={hist['loss'][-1]:.3f}")

    print(f"обучение: {time.time() - t0:.1f}с, шагов среды={global_step}, "
          f"переходов в буфере={len(replay)}, градиентных={grad_steps}")
    online.eval()
    return online, target, hist

## 9. Запуск обучения

`N_EPISODES = 300` × `EPISODE_STEPS = 900` — это 270 тыс. шагов среды и
~1.08 млн переходов в буфере (по 4 агента на шаг), из них ~135 тыс.
градиентных шагов. На CPU это порядка 15 минут, на GPU быстрее; если нужно
быстрее — уменьшите `N_EPISODES` (например, до 120) или `EPISODE_STEPS`.

Если greedy-политика в п. 10 не обгоняет `Fixed-timer`, увеличьте
`N_EPISODES`/`EPS_DECAY_STEPS`: сеть общая на 4 узла, и ей нужно больше
данных, чем одиночному агенту.

In [ ]:
# ============================================================
# 9. Запуск обучения
# ============================================================
online_net, target_net, history = train_agent(N_EPISODES)

In [ ]:
# ============================================================
# 9b. Графики обучения
# ============================================================
def moving_average(x, k=10):
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


fig, axes = plt.subplots(3, 2, figsize=(14, 11))
ax = axes.ravel()

ax[0].plot(history["reward"], alpha=0.3, label="по эпизодам")
ax[0].plot(moving_average(history["reward"]), label="среднее (10)")
ax[0].set_title("Суммарная награда сети за эпизод")
ax[0].legend()

ax[1].plot(moving_average(history["queue"]))
ax[1].set_title("Средняя суммарная очередь, авто")

ax[2].plot(moving_average(history["served"]), label="обслужено на узлах")
ax[2].plot(moving_average(history["completed"]), label="выехало из сети")
ax[2].set_title("Throughput, авто/эпизод")
ax[2].legend()

ax[3].plot(moving_average(history["wait"]))
ax[3].set_title("Среднее ожидание обслуженных, с")

ax[4].plot(np.asarray(history["starvation"]) * 100, label="starvation, %")
ax[4].plot(moving_average(history["switches"]), label="переключений/эпизод")
ax[4].set_title("Starvation и частота переключений")
ax[4].legend()

ax[5].plot(moving_average(history["loss"]), label="Huber loss")
ax[5].plot(history["epsilon"], alpha=0.7, label="epsilon")
ax[5].set_title("Loss и epsilon")
ax[5].legend()

for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Оценка: DQN vs baselines

Все политики играют на одинаковых сидах прибытий (`seed=10000+i`), по
`EVAL_EPISODES` эпизодов, метрики усредняются по сети. Кроме fixed-timer и
random — две эвристики: `LongestWait` (после `MIN_GREEN` — фаза, где
дольше всех ждёт головная машина) и `Always-hold` (среда сама переключает по
`MAX_GREEN`). Вторая таблица — served по узлам: видно, не морит ли политика
голодом конкретный перекрёсток, что для сети важнее среднего по всем.

In [ ]:
# ============================================================
# 10. Политики и оценка
# ============================================================
class FixedTimerPolicy:
    """Фиксированный цикл: FIXED_GREEN секунд на каждую фазу."""

    def __init__(self, green_time=FIXED_GREEN):
        self.green_time = green_time

    def act(self, node_env):
        if node_env.mode != "green":
            return ACTION_STAY
        return (ACTION_SWITCH if node_env.green_elapsed >= self.green_time
                else ACTION_STAY)


class LongestWaitPolicy:
    """Переключаться, если у следующей фазы голова ждёт заметно дольше.

    Порядок фаз фиксирован, поэтому сравниваем только текущую фазу и
    следующую. Считать просто длину очереди здесь нельзя: спрос на левый
    поворот вчетверо меньше, чем на прямой, и жадная политика по длине
    очереди никогда не отдала бы левую фазу — левые машины копились бы на
    голове полосы и блокировали движение. Поэтому сравниваем *ожидание*
    тех головных машин, которые фаза реально может обслужить.
    """

    def __init__(self, min_green=MIN_GREEN, hysteresis=5.0):
        self.min_green = min_green
        self.hysteresis = hysteresis

    def act(self, node_env, t):
        if node_env.mode != "green":
            return ACTION_STAY

        def head_wait(p):
            w = 0.0
            for a in PHASE_ACTIVE[p]:
                q = node_env.queues[a]
                if q and q[0].maneuver in PHASE_MANEUVERS[p]:
                    w = max(w, (t - q[0].arrival_step) * DT)
            return w

        if node_env.green_elapsed >= MAX_GREEN:
            return ACTION_SWITCH
        if node_env.green_elapsed < self.min_green:
            return ACTION_STAY
        nxt = (node_env.phase + 1) % N_PHASES
        return (ACTION_SWITCH
                if head_wait(nxt) > head_wait(node_env.phase) + self.hysteresis
                else ACTION_STAY)


class AlwaysHoldPolicy:
    """Всегда «держать»: среда сама переключает фазы по MAX_GREEN."""

    def act(self, node_env):
        return ACTION_STAY


def run_episode(env, policy):
    states = env.reset()
    total_r, q_sum, starv, switches, steps = 0.0, 0.0, 0.0, 0, 0
    while True:
        masks = env.action_masks()
        actions = {n: policy(n, env, states[n], masks[n]) for n in NODES}
        states, rewards, done, infos = env.step(actions)
        total_r += float(sum(rewards.values()))
        q_sum += sum(info["queue"] for info in infos.values())
        starv += (sum(1 for info in infos.values() if info["starving"] > 0)
                  / len(NODES))
        switches += sum(info["switched"] for info in infos.values())
        steps += 1
        if done:
            break
    served_total = sum(nd.served for nd in env.nodes.values())
    wait_sum = sum(nd.served_wait_sum for nd in env.nodes.values())
    out = {"reward": total_r, "served": served_total,
           "completed": env.completed, "avg_queue": q_sum / steps,
           "avg_wait": wait_sum / max(1, served_total),
           "starvation": starv / steps, "dropped": env.dropped_total,
           "switches": switches}
    for i, n in enumerate(NODES):
        out[f"srv_{i}"] = env.nodes[n].served
    return out


def evaluate(policy, episodes=EVAL_EPISODES, seed0=10_000):
    rows = [run_episode(TrafficNetworkEnv(seed=seed0 + i), policy)
            for i in range(episodes)]
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}


eval_rng = np.random.default_rng(7)


def dqn_policy(net):
    def act(node, env, state, mask):
        return int(select_actions(net, state[None], mask[None], 0.0,
                                  eval_rng)[0])
    return act


fixed_timer = FixedTimerPolicy(FIXED_GREEN)
longest_wait = LongestWaitPolicy()
always_hold = AlwaysHoldPolicy()

policies = {
    "DQN (greedy)": dqn_policy(online_net),
    f"Fixed-timer {FIXED_GREEN}с": lambda n, e, s, m: fixed_timer.act(e.nodes[n]),
    "LongestWait": lambda n, e, s, m: longest_wait.act(e.nodes[n], e.t),
    "Always-hold": lambda n, e, s, m: always_hold.act(e.nodes[n]),
    "Random": lambda n, e, s, m: int(eval_rng.choice(np.flatnonzero(m))),
}
results = {name: evaluate(pol) for name, pol in policies.items()}

print(f"{'политика':<20}{'reward':>10}{'served':>8}{'выехало':>9}"
      f"{'очередь':>9}{'ожид,с':>8}{'starv,%':>9}{'отказы':>8}{'switch':>8}")
for name, r in results.items():
    print(f"{name:<20}{r['reward']:>10.1f}{r['served']:>8.1f}"
          f"{r['completed']:>9.1f}{r['avg_queue']:>9.2f}{r['avg_wait']:>8.1f}"
          f"{r['starvation'] * 100:>9.1f}{r['dropped']:>8.1f}"
          f"{r['switches']:>8.1f}")

print(f"\n{'политика':<20}{'СЗ':>8}{'СВ':>8}{'ЮЗ':>8}{'ЮВ':>8}")
for name, r in results.items():
    print(f"{name:<20}" + "".join(f"{r[f'srv_{i}']:>8.1f}"
                                  for i in range(len(NODES))))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, key, title in zip(
        axes,
        ["completed", "avg_queue", "avg_wait", "starvation"],
        ["Выехало из сети, авто", "Средняя очередь, авто",
         "Ср. ожидание, с", "Доля шагов с голоданием"]):
    vals = [results[n][key] * (100 if key == "starvation" else 1)
            for n in results]
    ax.bar(list(results), vals)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()

# ---------- диагностика greedy: зелёное время и действия по узлам ----------
diag_env = TrafficNetworkEnv(seed=777_777)
diag_states = diag_env.reset()
phase_time = {n: np.zeros(N_PHASES) for n in NODES}
actions_hist = {n: np.zeros(N_ACTIONS, dtype=int) for n in NODES}
diag_switches = 0
diag_snap = None

while True:
    masks = diag_env.action_masks()
    actions = {}
    for n in NODES:
        a = int(select_actions(online_net, diag_states[n][None],
                               masks[n][None], 0.0, eval_rng)[0])
        actions[n] = a
        actions_hist[n][a] += 1
    for n, nd in diag_env.nodes.items():
        if nd.mode == "green":
            phase_time[n][nd.phase] += 1
    diag_states, _, done, infos = diag_env.step(actions)
    diag_switches += sum(info["switched"] for info in infos.values())
    if diag_env.t == 450:
        diag_snap = diag_env.snapshot()
    if done:
        break

print("\n--- диагностика greedy-политики ---")
print(f"переключений: {diag_switches} | выехало: {diag_env.completed} | "
      f"отказы: {diag_env.dropped_total}")
for n in NODES:
    share = phase_time[n] / max(1.0, phase_time[n].sum()) * 100
    print(f"{NODE_NAMES[n]}: served={diag_env.nodes[n].served:3d} | "
          "зелёное время " + "/".join(f"{PHASE_SHORT[p]} {share[p]:4.1f}%"
                                         for p in PHASES) +
          f" | держать {actions_hist[n][ACTION_STAY]}"
          f"/switch {actions_hist[n][ACTION_SWITCH]}")
ascii_summary(diag_env)
if diag_snap is not None:
    fig, ax = plt.subplots(figsize=(7.2, 7.6))
    draw_snapshot(diag_snap, ax=ax, legend=True)
    fig.tight_layout()
    plt.show()

## 11. Анимация работы сети

`animate_network(policy)` проигрывает эпизод и рисует каждый шаг: цвет
дорожки у стоп-линии — сигнал, блоки в полосе — очередь (тёмные едут
прямо, оранжевые — направо/ППД, фиолетовые — налево), в квадрате узла —
стрелка текущей фазы и число машин.
По умолчанию играет `LongestWait`; можно передать `dqn_policy(online_net)`.

Анимация собирается в HTML (`to_jshtml`), поэтому её видно в ноутбуке без
ffmpeg. 120 кадров рендерятся ~10–20 с; уменьшите `steps`, если тяжело.

In [ ]:
# ============================================================
# 11. Анимация
# ============================================================
def animate_network(policy=None, steps=120, interval=120, seed=2024):
    if policy is None:
        heuristic = LongestWaitPolicy()
        policy = lambda n, e, s, m: heuristic.act(e.nodes[n], e.t)
    env = TrafficNetworkEnv(seed=seed)
    env.reset()
    fig, ax = plt.subplots(figsize=(6.8, 7.2))

    def frame(k):
        ax.clear()
        draw_snapshot(env.snapshot(), ax=ax, legend=True)
        masks = env.action_masks()
        cur = env.state()
        env.step({n: policy(n, env, cur[n], masks[n]) for n in NODES})

    anim = FuncAnimation(fig, frame, frames=steps, interval=interval,
                         repeat=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


animate_network(steps=120)

## 12. Сохранение и загрузка модели

В чекпоинт кладём `state_dict` и размерности, чтобы восстановить сеть без
знания всего кода обучения. Модель обучена на 4 узлах с parameter sharing,
поэтому загруженная сеть сразу пригодна для любого из них.

In [ ]:
# ============================================================
# 12. Сохранение / загрузка
# ============================================================
checkpoint = {
    "model_state": online_net.state_dict(),
    "state_dim": STATE_DIM,
    "n_actions": N_ACTIONS,
    "hidden": HIDDEN,
    "nodes": NODES,
    "episode": N_EPISODES,
}
torch.save(checkpoint, MODEL_PATH)
print("сохранено:", MODEL_PATH)


def load_agent(path=MODEL_PATH):
    try:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:      # старые версии torch без weights_only
        ckpt = torch.load(path, map_location=DEVICE)
    net = DQNNet(ckpt["state_dim"], ckpt["n_actions"], ckpt["hidden"]).to(DEVICE)
    net.load_state_dict(ckpt["model_state"])
    net.eval()
    return net


loaded_net = load_agent()
check = evaluate(dqn_policy(loaded_net), episodes=2)
print(f"загруженная модель: выехало={check['completed']:.1f}, "
      f"served={check['served']:.1f}, очередь={check['avg_queue']:.2f}, "
      f"ожидание={check['avg_wait']:.1f}с, "
      f"starvation={check['starvation'] * 100:.1f}%")

## 13. Валидация в SUMO и сравнение со встроенными контроллерами

Своя среда — учебная модель. Проверяем политику в микроскопическом
симуляторе SUMO: та же сеть 2x2 (4 перекрёстка, по одной полосе на подход,
13.9 м/с), те же пропорции спроса (`RATE_WEIGHT`, `MANEUVER_P`), та же логика
нашего контроллера (`MIN_GREEN`/`MAX_GREEN`, жёлтый + красный) и тот же вход
сети: очереди считаем по стоящим машинам (скорость < 0.1 м/с) с разбивкой по
манёврам, ожидание головной — `waitingTime`, 23 числа на узел.

Заодно сравниваем со **встроенными** контроллерами SUMO, которые генерирует
`netconvert`:

* `static` — фиксированный цикл из коробки: 2 фазы (С-Ю и З-В), левый
  поворот разрешённый (permissive) и едет вместе со встречным прямым;
* `actuated` — актуатор: min/max green, gap-out по детекторам, продлевает
  фазу, пока под ней есть машины;
* `delay_based` — адаптивный выбор фазы по накопленной задержке;
* варианты `4ф` — те же типы, но с `--tls.layout incoming --tls.no-mixed`:
  каждая фаза обслуживает один подход целиком, левые защищены. Это ближе к
  нашей 4-фазной схеме, поэтому сравнение честнее.



In [26]:
# ============================================================
# 13. Валидация в SUMO (та же сеть, тот же контроллер, та же модель)
# ============================================================
import os
import subprocess
import xml.etree.ElementTree as ET

import sumolib
import traci

SUMO_DIR = "sumo_valid"
SUMO_RATE = 0.03


def find_sumo_bin(name):
    exe = name + ".exe" if os.name == "nt" else name
    cands = []
    home = os.environ.get("SUMO_HOME")
    if home:
        cands.append(os.path.join(home, "bin", name))
        cands.append(os.path.join(home, "bin", exe))
    try:
        cands.append(sumolib.checkBinary(name))
    except Exception:
        pass
    try:
        import sumo as sumo_pkg
        cands.append(os.path.join(os.path.dirname(sumo_pkg.__file__),
                                  "bin", exe))
    except Exception:
        pass
    for base in ("C:/Program Files/Eclipse/Sumo/bin",
                 "C:/Program Files (x86)/Eclipse/Sumo/bin",
                 "C:/sumo/bin", "/usr/local/bin", "/usr/bin"):
        cands.append(os.path.join(base, exe))
    for c in cands:
        if c and os.path.isfile(c):
            return c
    raise RuntimeError(
        "Не найден %s. Поставьте SUMO (pip install eclipse-sumo sumolib "
        "traci) или задайте SUMO_HOME, либо выберите kernel tf_gpu." % exe)


SUMO = find_sumo_bin("sumo")
NETCONVERT = find_sumo_bin("netconvert")
DUAROUTER = find_sumo_bin("duarouter")

S_GRID = {(0, 0): (0.0, 200.0), (0, 1): (200.0, 200.0),
          (1, 0): (0.0, 0.0), (1, 1): (200.0, 0.0)}
S_NID = {(0, 0): "J00", (0, 1): "J01", (1, 0): "J10", (1, 1): "J11"}
S_EXT = {"J00": {"N": (0.0, 400.0), "W": (-200.0, 200.0)},
         "J01": {"N": (200.0, 400.0), "E": (400.0, 200.0)},
         "J10": {"S": (0.0, -200.0), "W": (-200.0, 0.0)},
         "J11": {"S": (200.0, -200.0), "E": (400.0, 0.0)}}
S_EXIT = {
    ("N", "left"): "J01_XJ01E", ("N", "straight"): "J10_XJ10S",
    ("N", "right"): "J00_XJ00W",
    ("S", "left"): "J10_XJ10W", ("S", "straight"): "J00_XJ00N",
    ("S", "right"): "J11_XJ11E",
    ("E", "left"): "J00_XJ00N", ("E", "straight"): "J01_XJ01E",
    ("E", "right"): "J10_XJ10S",
    ("W", "left"): "J11_XJ11S", ("W", "straight"): "J10_XJ10W",
    ("W", "right"): "J01_XJ01N",
}
S_ENTRY = {"J00": {"N": "XJ00N_J00", "W": "XJ00W_J00"},
           "J01": {"N": "XJ01N_J01", "E": "XJ01E_J01"},
           "J10": {"S": "XJ10S_J10", "W": "XJ10W_J10"},
           "J11": {"S": "XJ11S_J11", "E": "XJ11E_J11"}}
S_DIRS = {"N": (0.0, 1.0), "S": (0.0, -1.0), "E": (1.0, 0.0), "W": (-1.0, 0.0)}


def sumo_build_network(tls_type="static", layout=None):
    os.makedirs(SUMO_DIR, exist_ok=True)
    nodes = ['<nodes>']
    for rc, (x, y) in S_GRID.items():
        nodes.append('  <node id="%s" x="%.1f" y="%.1f" type="traffic_light"/>'
                     % (S_NID[rc], x, y))
    for j, arms in S_EXT.items():
        for a, (x, y) in arms.items():
            nodes.append('  <node id="X%s%s" x="%.1f" y="%.1f" '
                         'type="priority"/>' % (j, a, x, y))
    nodes.append('</nodes>')
    with open(os.path.join(SUMO_DIR, "nodes.nod.xml"), "w", encoding="utf-8") as f:
        f.write("\n".join(nodes) + "\n")

    edges = ['<edges>']
    pairs = [((0, 0), (0, 1)), ((1, 0), (1, 1)),
             ((0, 0), (1, 0)), ((0, 1), (1, 1))]
    for a, b in pairs:
        for u, v in ((a, b), (b, a)):
            edges.append('  <edge id="%s_%s" from="%s" to="%s" numLanes="1" '
                         'speed="13.9"/>' % (S_NID[u], S_NID[v], S_NID[u],
                                             S_NID[v]))
    for j, arms in S_EXT.items():
        for a in arms:
            edges.append('  <edge id="X%s%s_%s" from="X%s%s" to="%s" '
                         'numLanes="1" speed="13.9"/>' % (j, a, j, j, a, j))
            edges.append('  <edge id="%s_X%s%s" from="%s" to="X%s%s" '
                         'numLanes="1" speed="13.9"/>' % (j, j, a, j, j, a))
    edges.append('</edges>')
    with open(os.path.join(SUMO_DIR, "edges.edg.xml"), "w", encoding="utf-8") as f:
        f.write("\n".join(edges) + "\n")

    tag = tls_type if layout is None else "%s_%s" % (tls_type, layout)
    net = os.path.join(SUMO_DIR, "net_%s.net.xml" % tag)
    extra = [] if layout is None else ["--tls.layout", layout,
                                       "--tls.no-mixed", "true"]
    r = subprocess.run([NETCONVERT, "-n", os.path.join(SUMO_DIR, "nodes.nod.xml"),
                        "-e", os.path.join(SUMO_DIR, "edges.edg.xml"),
                        "-o", net, "--no-turnarounds", "true",
                        "--junctions.corner-detail", "0",
                        "--rectangular-lane-cut", "false",
                        "--tls.default-type", tls_type] + extra,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:])
    return net


def sumo_write_trips(horizon=EPISODE_STEPS, seed=7):
    rng = np.random.default_rng(seed)
    rows = []
    k = 0
    for j, arms in S_ENTRY.items():
        for a, edge in arms.items():
            for man, p in MANEUVER_P.items():
                lam = SUMO_RATE * RATE_WEIGHT[a] * p
                for _ in range(rng.poisson(lam * horizon)):
                    k += 1
                    rows.append((rng.uniform(0, horizon - 1), k, edge,
                                 S_EXIT[(a, man)]))
    rows.sort()
    trips = ['<routes>']
    for depart, kid, edge, to in rows:
        trips.append('  <trip id="v%d" depart="%.2f" from="%s" '
                     'to="%s"/>' % (kid, depart, edge, to))
    trips.append('</routes>')
    path = os.path.join(SUMO_DIR, "trips.trips.xml")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(trips) + "\n")
    return path


def sumo_build_routes(net, trips):
    routes = os.path.join(SUMO_DIR, "routes.rou.xml")
    r = subprocess.run([DUAROUTER, "-n", net, "--route-files", trips,
                        "-o", routes, "--ignore-errors", "true",
                        "--no-warnings", "true"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:])
    return routes


class SumoNode:
    """Контроллер перекрёстка: фазы, маска, вектор состояния из TraCI."""

    def __init__(self, tls_id, net):
        self.tls = tls_id
        self.phase = 0
        self.mode = "green"
        self.elapsed = 0
        self.timer = 0
        self.pending = None
        self.green = "r" * len(traci.trafficlight.getControlledLinks(tls_id))
        self.links = []
        self.edges = {a: None for a in APPROACHES}
        for i, group in enumerate(traci.trafficlight.getControlledLinks(
                tls_id)):
            for link in group:
                el = net.getEdge(traci.lane.getEdgeID(link[0]))
                to = net.getEdge(traci.lane.getEdgeID(link[1]))
                app = OPPOSITE[self._dir_edge(el)]
                man = self._maneuver(el, to)
                self.links.append((i, app, man))
                if self.edges[app] is None:
                    self.edges[app] = el.getID()

    @staticmethod
    def _dir_edge(e):
        fx, fy = e.getFromNode().getCoord()
        tx, ty = e.getToNode().getCoord()
        dx, dy = tx - fx, ty - fy
        if abs(dx) > abs(dy):
            return "E" if dx > 0 else "W"
        return "N" if dy > 0 else "S"

    def _maneuver(self, edge, to_edge):
        d1 = S_DIRS[self._dir_edge(edge)]
        d2 = S_DIRS[self._dir_edge(to_edge)]
        cross = d1[0] * d2[1] - d1[1] * d2[0]
        if abs(cross) < 0.5:
            return "straight"
        return "left" if cross > 0 else "right"

    def _queues(self):
        out = {a: [] for a in APPROACHES}
        for a, e in self.edges.items():
            if e is None:
                continue
            for vid in traci.edge.getLastStepVehicleIDs(e):
                if traci.vehicle.getSpeed(vid) > 0.1:
                    continue
                route = traci.vehicle.getRoute(vid)
                idx = traci.vehicle.getRouteIndex(vid)
                if idx + 1 >= len(route):
                    continue
                nxt = route[idx + 1]
                try:
                    man = self._maneuver(self._net.getEdge(e),
                                         self._net.getEdge(nxt))
                except KeyError:
                    continue
                out[a].append((man, traci.vehicle.getWaitingTime(vid),
                               traci.vehicle.getLanePosition(vid)))
        for a in out:
            out[a].sort(key=lambda x: -x[2])
        return out

    def state(self, t):
        qs = self._queues()
        s = np.zeros(STATE_DIM, dtype=np.float32)
        i = 0
        for a in APPROACHES:
            items = qs[a]
            s[i + 0] = sum(1 for m, _, _ in items
                           if m == "straight") / QUEUE_CAPACITY
            s[i + 1] = sum(1 for m, _, _ in items if m == "left") \
                / QUEUE_CAPACITY
            s[i + 2] = sum(1 for m, _, _ in items if m == "right") \
                / QUEUE_CAPACITY
            if items:
                s[i + 3] = min(items[0][1] / MAX_WAIT_NORM, 1.5)
            i += 4
        s[i + self.phase] = 1.0
        i += N_PHASES
        s[i + 0] = min(self.elapsed / MAX_GREEN, 1.0)
        s[i + 1] = 0.0 if self.mode == "green" else 1.0
        s[i + 2] = (self.timer / (YELLOW_TIME + ALL_RED_TIME)
                    if self.mode != "green" else 0.0)
        return s

    def mask(self):
        m = np.zeros(N_ACTIONS, dtype=bool)
        m[ACTION_STAY] = True
        if self.mode == "green":
            m[ACTION_SWITCH] = self.elapsed >= MIN_GREEN
            if self.elapsed >= MAX_GREEN:
                m[ACTION_STAY] = False
        return m

    def apply(self, action):
        if self.mode != "green":
            return
        if action == ACTION_SWITCH or self.elapsed >= MAX_GREEN:
            if self.elapsed >= MIN_GREEN or self.elapsed >= MAX_GREEN:
                self.mode = "yellow"
                self.timer = YELLOW_TIME
                self.pending = (self.phase + 1) % N_PHASES

    def tick(self):
        if self.mode == "green":
            self.elapsed += 1
        elif self.mode == "yellow":
            self.timer -= 1
            if self.timer <= 0:
                self.mode = "all_red"
                self.timer = ALL_RED_TIME
        else:
            self.timer -= 1
            if self.timer <= 0:
                self.mode = "green"
                self.phase = self.pending
                self.pending = None
                self.elapsed = 0

    def signal(self):
        if self.mode == "yellow":
            return self.green.replace("G", "y").replace("g", "y")
        if self.mode == "all_red":
            return "r" * len(self.green)
        state = ["r"] * len(self.green)
        for i, app, man in self.links:
            if app in PHASE_ACTIVE[self.phase] and \
                    man in PHASE_MANEUVERS[self.phase]:
                state[i] = "G"
        self.green = "".join(state)
        return self.green


class SumoFixed:
    def __init__(self, green):
        self.green = green

    def __call__(self, state, mask, elapsed, t):
        return ACTION_SWITCH if elapsed >= self.green else ACTION_STAY


class SumoRandom:
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def __call__(self, state, mask, elapsed, t):
        return int(self.rng.choice(np.flatnonzero(mask)))


class SumoLongest:
    def __init__(self, node):
        self.nd = node

    def __call__(self, state, mask, elapsed, t):
        nd = self.nd
        if nd.mode != "green":
            return ACTION_STAY

        def hw(p):
            w = 0.0
            for a in PHASE_ACTIVE[p]:
                items = nd._queues()[a]
                if items and items[0][0] in PHASE_MANEUVERS[p]:
                    w = max(w, items[0][1])
            return w

        if nd.elapsed >= MAX_GREEN:
            return ACTION_SWITCH
        if nd.elapsed < MIN_GREEN:
            return ACTION_STAY
        nxt = (nd.phase + 1) % N_PHASES
        return ACTION_SWITCH if hw(nxt) > hw(nd.phase) + 5 else ACTION_STAY


class SumoDqn:
    def __init__(self, net):
        self.net = net
        self.rng = np.random.default_rng(0)

    def __call__(self, state, mask, elapsed, t):
        arr = np.asarray(state, dtype=np.float32).reshape(1, -1)
        msk = np.asarray(mask, dtype=bool).reshape(1, -1)
        return int(select_actions(self.net, arr, msk, 0.0, self.rng)[0])


def sumo_run(net_path, route_path, policy_factory=None,
             horizon=EPISODE_STEPS, seed=1):
    net = sumolib.net.readNet(net_path)
    tripinfo = os.path.join(SUMO_DIR, "tripinfo.xml")
    if os.path.exists(tripinfo):
        os.remove(tripinfo)
    cmd = [SUMO, "-n", net_path, "-r", route_path,
           "--begin", "0", "--end", str(horizon), "--step-length", "1",
           "--seed", str(seed), "--no-step-log", "true",
           "--no-warnings", "true", "--duration-log.statistics", "false",
           "--tripinfo-output", tripinfo]
    traci.start(cmd)
    all_edges = [e.getID() for e in net.getEdges()]
    arrived = halting_sum = wait_sum = veh_samples = steps = 0
    try:
        nodes = {}
        if policy_factory is not None:
            for tid in traci.trafficlight.getIDList():
                nodes[tid] = SumoNode(tid, net)
                nodes[tid]._net = net
        for t in range(horizon):
            for tid, nd in nodes.items():
                st = nd.state(t)
                act = policy_factory(tid, nd)(st, nd.mask(), nd.elapsed, t)
                nd.apply(int(act))
                nd.tick()
                traci.trafficlight.setRedYellowGreenState(tid, nd.signal())
            traci.simulationStep()
            steps += 1
            arrived += traci.simulation.getArrivedNumber()
            halting_sum += sum(traci.edge.getLastStepHaltingNumber(e)
                               for e in all_edges)
            for v in traci.vehicle.getIDList():
                wait_sum += traci.vehicle.getWaitingTime(v)
                veh_samples += 1
            if traci.simulation.getMinExpectedNumber() == 0:
                break
        teleports = traci.simulation.getStartingTeleportNumber()
        duration = traci.simulation.getTime()
    finally:
        traci.close()

    trips = []
    if os.path.exists(tripinfo):
        for tr in ET.parse(tripinfo).getroot():
            trips.append((float(tr.get("duration")),
                          float(tr.get("waitingTime"))))
    dur = np.array([x[0] for x in trips]) if trips else np.zeros(1)
    wt = np.array([x[1] for x in trips]) if trips else np.zeros(1)
    return {"arrived": arrived, "sim_time": duration,
            "halting_mean": halting_sum / max(1, steps),
            "wait_per_veh": wait_sum / max(1, veh_samples),
            "trip_duration": float(dur.mean()),
            "trip_wait": float(wt.mean()),
            "teleports": teleports, "finished_trips": len(trips)}


sumo_nets = {t: sumo_build_network(t)
             for t in ("static", "actuated", "delay_based")}
sumo_nets["static_incoming"] = sumo_build_network("static", "incoming")
sumo_nets["actuated_incoming"] = sumo_build_network("actuated", "incoming")
sumo_trips = sumo_write_trips()
sumo_routes = sumo_build_routes(sumo_nets["static"], sumo_trips)
n_trips = open(sumo_routes, encoding="utf-8").read().count("<vehicle")
print("маршрутов: %d | горизонт: %d с | спрос: %.2f авто/с"
      % (n_trips, EPISODE_STEPS, n_trips / EPISODE_STEPS))

sumo_runs = (
    ("SUMO static", sumo_nets["static"], None),
    ("SUMO actuated", sumo_nets["actuated"], None),
    ("SUMO delay_based", sumo_nets["delay_based"], None),
    ("SUMO static 4ф", sumo_nets["static_incoming"], None),
    ("SUMO actuated 4ф", sumo_nets["actuated_incoming"], None),
    ("FixedTimer(12)", sumo_nets["static"], lambda tid, nd: SumoFixed(12)),
    ("LongestWait", sumo_nets["static"], lambda tid, nd: SumoLongest(nd)),
    ("Random", sumo_nets["static"], lambda tid, nd: SumoRandom(0)),
    ("DQN", sumo_nets["static"], lambda tid, nd: SumoDqn(online_net)),
)
sumo_rows = {}
print("%-17s %9s %11s %10s %9s %10s" % ("политика", "завершено",
      "ожидание, с", "тормозящих", "поездка", "телепорты"))
for name, net_p, factory in sumo_runs:
    r = sumo_run(net_p, sumo_routes, factory, seed=1)
    sumo_rows[name] = r
    print("%-17s %9d %11.1f %10.1f %9.1f %10d"
          % (name, r["arrived"], r["wait_per_veh"], r["halting_mean"],
             r["trip_duration"], r["teleports"]))

best = max(sumo_rows, key=lambda k: sumo_rows[k]["arrived"])
print("больше всех доехало:", best)

маршрутов: 309 | горизонт: 900 с | спрос: 0.34 авто/с
политика          завершено ожидание, с тормозящих   поездка  телепорты
SUMO static             254         4.7       13.1     115.5          0
SUMO actuated           270         0.6        2.9      78.9          0
SUMO delay_based        275         0.8        2.9      78.7          0
SUMO static 4ф          238        13.0       23.9     148.3          0
SUMO actuated 4ф        258         4.2       13.1     112.6          0
FixedTimer(12)          232         8.3       24.3     153.2          0
LongestWait             204        14.2       38.4     185.5          0
Random                  155        31.6       62.0     213.2          0
DQN                     233         5.8       22.6     149.0          0
больше всех доехало: SUMO delay_based
